# Stage 2 - Instruction Fine-Tuning (SFT)

**Goal:** teach the model to answer questions - map a natural-language question to either
the correct **table name** (schema discovery) or a valid **SQL query** (support queries).

Data: `ecomm-db-instruction` on the Hugging Face Hub (215 `{instruction, response}` examples).

In [ ]:
# Install Unsloth (needs a CUDA GPU). If Colab prompts to restart after install, do it.
%%capture
!pip install --upgrade unsloth unsloth_zoo
# Ensure a MODERN TRL (provides SFTConfig/DPOConfig + processing_class) that matches new transformers.
!pip install --upgrade "trl>=0.13.1"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

## 1. Config

In [ ]:
# --- Datasets live on the Hugging Face Hub (pushed via scripts/push_to_hf.py) ---
HF_USER = "Rajesh507"   # <-- your Hugging Face username
DS_NONINSTRUCT = f"{HF_USER}/ecomm-db-noninstruct"
DS_INSTRUCTION = f"{HF_USER}/ecomm-db-instruction"
DS_PREFERENCE  = f"{HF_USER}/ecomm-db-preference"

# If the datasets are PRIVATE, log in first (needs a read token):
# from huggingface_hub import login; login()

In [ ]:
# Persist trained adapters to the Hugging Face Hub (no Google Drive needed).
from huggingface_hub import login
login()   # paste a WRITE token: https://huggingface.co/settings/tokens

ADAPTER_STAGE1 = f"{HF_USER}/ecomm-db-stage1-noninstruct"
ADAPTER_STAGE2 = f"{HF_USER}/ecomm-db-stage2-sft"
ADAPTER_STAGE3 = f"{HF_USER}/ecomm-db-stage3-dpo"

# Merged 16-bit models that chain the stages: Stage N merges its LoRA into the
# base, and Stage N+1 loads that merged model and adds a FRESH, fully-trainable
# LoRA on top (bootcamp Class-22's "merge then add new adapter" approach).
MERGED_STAGE1 = f"{HF_USER}/ecomm-db-stage1-merged"
MERGED_STAGE2 = f"{HF_USER}/ecomm-db-stage2-merged"
print("Adapters will be pushed under:", HF_USER)

## 2. Load model
Continue from the merged Stage-1 model (domain-adapted), then add a fresh LoRA. Set `START_MODEL = MODEL_NAME` to skip Stage 1.

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B"   # Coder variant is stronger at SQL. Alt: "unsloth/Qwen2.5-1.5B"
MAX_SEQ_LEN = 2048

# Stage 2 starting point (chained pipeline):
#   START_MODEL = MERGED_STAGE1  -> continue from Stage-1 domain adaptation (the taught flow)
#   START_MODEL = MODEL_NAME     -> skip Stage 1 and train on the plain base model
START_MODEL = MERGED_STAGE1

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = START_MODEL,   # merged Stage-1 model (see START_MODEL above)
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

In [ ]:
from unsloth import FastLanguageModel

# If the loaded model ALREADY has a LoRA adapter (i.e. we continued from a
# previous stage's adapter), keep training THAT adapter instead of adding a new,
# conflicting one. Otherwise (a fresh base model) attach a new LoRA adapter.
if getattr(model, "peft_config", None):
    print("Model already has a LoRA adapter - continuing to train it.")
    try:
        FastLanguageModel.for_training(model)
    except Exception:
        pass
    # A loaded adapter is often frozen (inference mode); re-enable grads so
    # training actually updates it.
    for _n, _p in model.named_parameters():
        if "lora_" in _n:
            _p.requires_grad_(True)
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,                       # LoRA rank
        lora_alpha = 16,              # scaling
        lora_dropout = 0,             # 0 is optimized in Unsloth
        bias = "none",
        target_modules = ["q_proj","k_proj","v_proj","o_proj",
                          "gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

# Sanity: there MUST be trainable parameters, else trainer.train() does nothing
# and the model stays generic. This catches a frozen/misloaded adapter early.
model.print_trainable_parameters()
_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _trainable > 0, "No trainable parameters - adapter is frozen; training would do nothing."
print("Trainable params:", _trainable)

## 3. Load and format the instruction dataset

In [ ]:
# A single, consistent prompt template used across ALL three stages.
PROMPT = """Below is a question about the client e-commerce database schema. \
Write a response that correctly answers it, giving the exact table name(s) or a valid SQL query.

### Question:
{}

### Answer:
{}"""

In [ ]:
from datasets import load_dataset
EOS = tokenizer.eos_token

def format_examples(batch):
    return {"text": [PROMPT.format(i, r) + EOS
                     for i, r in zip(batch["instruction"], batch["response"])]}

ds = load_dataset(DS_INSTRUCTION, split="train").map(format_examples, batched=True)
print(ds)
print(ds[0]["text"])

## 4. Train (SFT)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,   # new TRL API (was 'tokenizer=')
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        dataset_num_proc = 2,
        packing = False,        # small dataset: keep one example per sequence
        padding_free = False,   # avoids the max_length/padding_free error when packing is off
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 10,  # small dataset: more passes needed for exact X_ schema recall
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage2",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()

## 5. Push the SFT adapter + merged model to the Hugging Face Hub

In [ ]:
model.push_to_hub(ADAPTER_STAGE2, token=True)
tokenizer.push_to_hub(ADAPTER_STAGE2, token=True)
print("Pushed SFT adapter to:", ADAPTER_STAGE2)

# Also push a MERGED 16-bit model so Stage 3 (DPO) can load it as its base and
# add a fresh LoRA on top.
model.push_to_hub_merged(MERGED_STAGE2, tokenizer, save_method="merged_16bit", token=True)
print("Pushed merged Stage-2 model to:", MERGED_STAGE2)

## 6. Inference after SFT

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=128):
    text = PROMPT.format(question, "")
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("### Answer:")[-1].strip()

for q in [
    "Which table stores product images?",
    "Give me a query to find unique orders for a customer.",
    "Find all shipments that have not been delivered.",
]:
    print("Q:", q); print("A:", ask(q)); print("-"*60)

In [ ]:
# --- Self-check: is the trained adapter ACTUALLY affecting the output? ---
# These questions are copied VERBATIM from the training set, so a correctly
# trained + loaded adapter must reproduce the domain-specific answer under
# greedy decoding. If this FAILS (generic Orders/shipments tables), the adapter
# is not active or training did not converge - fix that before trusting any
# other output from this model.
checks = [
    ("Give me a query to find unique orders for a customer.", ["X_Order", "DISTINCT", "order_id"]),
    ("Which table stores product images?",                    ["X_Product_Images"]),
    ("Where is shipment and tracking information stored?",    ["X_Shipment"]),
]

all_ok = True
for q, must_have in checks:
    ans = ask(q)
    ok = all(tok.lower() in ans.lower() for tok in must_have)
    all_ok = all_ok and ok
    print(f"[{'PASS' if ok else 'FAIL'}] {q}")
    print("   ->", ans.replace(chr(10), ' ')[:200])
    if not ok:
        print("   expected to contain:", must_have)

print("\nADAPTER ACTIVE + LEARNED:", all_ok)
assert all_ok, (
    "Verbatim training examples were NOT reproduced. The adapter is not applied "
    "or the model is undertrained. Check: (1) you ran the training cell in THIS "
    "session, (2) START_MODEL points at the Stage-1 adapter, (3) epochs are high "
    "enough, and re-run without restarting the runtime mid-way."
)

### Next
Proceed to `dpo_alignment.ipynb`.